# VGG16 for CIFAR-10 Classification

**Module:** CN-7023 Artificial Intelligence & Machine Vision  
**Institution:** University of East London

---

## Objectives

1. Implement VGG16 architecture
2. Apply transfer learning
3. Compare with ResNet18 and Custom CNN
4. Analyze deep network performance

---

In [ ]:
# === SETUP: Handle paths for both local and Colab ===import osimport pathlib# Auto-detect environment and set pathsif 'google.colab' in str(get_ipython()):    print("[COLAB] Running in Google Colab")    # In Colab, ensure we're in the right location    if not os.path.exists('/content/UEL-ai-assignment'):        %cd /content        !git clone https://github.com/sebastien15/UEL-ai-assignment.git    %cd /content/UEL-ai-assignment    BASE_PATH = '/content/UEL-ai-assignment'else:    print("[LOCAL] Running locally")    # Local environment    BASE_PATH = '.'# Set up pathsDATA_PATH = os.path.join(BASE_PATH, 'data')RESULTS_PATH = os.path.join(BASE_PATH, 'results')# Create directoriesos.makedirs(os.path.join(RESULTS_PATH, 'figures'), exist_ok=True)os.makedirs(os.path.join(RESULTS_PATH, 'checkpoints'), exist_ok=True)os.makedirs(os.path.join(RESULTS_PATH, 'logs'), exist_ok=True)print(f"[OK] Setup complete!")print(f"[PATH] Base path: {BASE_PATH}")print(f"[PATH] Data path: {DATA_PATH}")print(f"[PATH] Results path: {RESULTS_PATH}")

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
import time
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

torch.manual_seed(42)
np.random.seed(42)

## 2. Data Loading

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(224),  # VGG expects 224x224
    transforms.RandomCrop(224, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

trainset = torchvision.datasets.CIFAR10(root=DATA_PATH, train=True, download=True, transform=train_transform)
testset = torchvision.datasets.CIFAR10(root=DATA_PATH, train=False, download=True, transform=test_transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

print(f'Data loaded. Batch size: 64 (smaller due to VGG size)')

## 3. VGG16 Model

In [ ]:
print('Loading VGG16...')
model = models.vgg16(pretrained=True)

# Modify classifier
model.classifier[6] = nn.Linear(4096, 10)
model = model.to(device)

print(f'VGG16 loaded with {sum(p.numel() for p in model.parameters()):,} parameters')

## 4. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

num_epochs = 30
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in tqdm(loader, desc='Training'):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Evaluating'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

print('Training VGG16...')
for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    train_loss, train_acc = train_epoch(model, trainloader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, testloader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f'Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%')
    print(f'Test:  Loss={test_loss:.4f}, Acc={test_acc:.2f}%')
    
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), os.path.join(RESULTS_PATH, 'checkpoints', 'vgg16_best.pth'))
        print(f'[BEST] Saved! Accuracy: {best_acc:.2f}%')

print(f'\nBest accuracy: {best_acc:.2f}%')

## 5. Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['test_loss'], label='Test')
ax1.set_title('VGG16 - Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['test_acc'], label='Test')
ax2.set_title('VGG16 - Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'figures', 'vgg16_training.png'), dpi=150)
plt.show()

## Summary

- VGG16 is very deep (16 layers) and has many parameters (~138M)
- Requires larger input images (224x224)
- Achieves good accuracy but slower to train
- Comparison with other models in Notebook 05

**Next: Compare all models!** 🚀